# Módulo 2 · 06 · Evaluación, interpretabilidad y alertas

Todo sobre el conjunto de prueba (cuatro conductores de State Farm que el modelo no vio y las imágenes de otro origen asignadas a prueba), con los modelos guardados: no se reentrena nada.

In [ ]:
# parameters
DEMO = False
UMBRAL = None   # umbral de alerta; None = el de config/modulo2.yaml

In [ ]:
import sys
from pathlib import Path
RAIZ = Path.cwd().resolve().parents[1]
sys.path[:0] = [str(RAIZ), str(RAIZ / "modulo2_distraccion")]

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from common.seeds import fijar_semilla
import m2lib, m2exp

fijar_semilla()
cfg = m2exp.config(demo=DEMO)
CLASES = cfg["datos"]["clases"]
ETIQ = [m2lib.NOMBRES[c] for c in CLASES]
FIG, RES = m2lib.FIGURAS, m2lib.RESULTADOS
FIG.mkdir(exist_ok=True); RES.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight"})

In [ ]:
from PIL import Image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
s = "_demo" if DEMO else ""
prueba = m2lib.validas().query("particion == 'prueba'").reset_index(drop=True)
if DEMO:
    prueba = prueba.sample(frac=1, random_state=0).groupby("clase").head(cfg["demo"]["imagenes_por_clase"]).reset_index(drop=True)
y = np.array([CLASES.index(c) for c in prueba["clase"]])
P = {"Regresión logística": np.load(RES / f"prob_prueba_logistica{s}.npy"),
     "Red desde cero": np.load(RES / f"prob_prueba_cnn{s}.npy"),
     "Preentrenada + ajuste fino": np.load(RES / f"prob_prueba_transfer{s}.npy")}
tabla = pd.DataFrame({k: m2lib.metricas(y, p, CLASES) for k, p in P.items()}).T
tabla.round(3).to_csv(RES / "metricas_prueba.csv")
tabla[["exactitud", "f1_macro"] + [f"precision_{c}" for c in CLASES] + [f"sensibilidad_{c}" for c in CLASES]].round(3).T

## Por origen de la imagen

In [ ]:
sf = prueba["conductor"].notna().to_numpy()
filas = []
for k, p in P.items():
    for nombre, mask in [("State Farm", sf), ("otro origen", ~sf)]:
        if mask.sum():
            m = m2lib.metricas(y[mask], p[mask], CLASES)
            filas.append({"modelo": k, "origen": nombre, "imagenes": int(mask.sum()), "exactitud": m["exactitud"], "f1_macro": m["f1_macro"]})
por_origen = pd.DataFrame(filas).round(3); por_origen.to_csv(RES / "metricas_por_origen.csv", index=False)
por_origen

## Matriz de confusión del mejor modelo

In [ ]:
mejor = tabla["f1_macro"].idxmax()
pm = P[mejor]; pred = pm.argmax(1)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay(confusion_matrix(y, pred), display_labels=ETIQ).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=30)
ax.set_title(mejor)
fig.savefig(FIG / "matriz_confusion.png")

## Aciertos y errores con Grad-CAM

Grad-CAM (Selvaraju et al., 2017) pondera los mapas de la última capa convolucional por el promedio del gradiente de la puntuación de la clase predicha. Rojo = regiones que más empujan la predicción.

In [ ]:
fam = "transfer" if mejor.startswith("Pre") else "cnn"
red, info_m = m2exp.cargar_modelo(fam)
red = red.to(m2lib.dispositivo())
capa = m2lib.capa_gradcam(red, info_m["tipo"])
rng = np.random.default_rng(0)
bien = rng.choice(np.where(pred == y)[0], 10, replace=False)
mal = rng.choice(np.where(pred != y)[0], min(10, int((pred != y).sum())), replace=False)

def panel(indices, titulo, archivo):
    fig, axs = plt.subplots(4, 5, figsize=(15, 9.2))
    for k, i in enumerate(indices):
        im = Image.open(m2lib.PROC / "imagenes" / prueba.archivo[i]).convert("RGB")
        x = m2lib.a_tensor(im)[0]
        mapa, _ = m2lib.gradcam(red, capa, x, int(pred[i]))
        a, b = axs[2 * (k // 5), k % 5], axs[2 * (k // 5) + 1, k % 5]
        a.imshow(im); a.axis("off")
        a.set_title(f"real: {m2lib.NOMBRES[CLASES[y[i]]]}\npred: {m2lib.NOMBRES[CLASES[pred[i]]]} ({pm[i].max():.2f})", fontsize=8)
        b.imshow(im); b.imshow(mapa, cmap="jet", alpha=.45); b.axis("off")
    for ax in axs.flat[2 * len(indices):]:
        ax.axis("off")
    fig.suptitle(titulo); fig.savefig(FIG / archivo)

panel(bien, f"10 imágenes bien clasificadas ({mejor})", "gradcam_aciertos.png")
panel(mal, f"10 imágenes mal clasificadas ({mejor})", "gradcam_errores.png")

## Política de alertas

Sin alerta: conducción segura y giro (alcanzar algo atrás es breve y no involucra el teléfono; ver el reporte). Con alerta: escribir, hablar por teléfono y otras actividades. Alerta si la suma de probabilidades de las clases con alerta supera el umbral.

In [ ]:
A = cfg["alertas"]
umbrales = np.round(np.arange(0.1, 0.951, 0.05), 2)
filas = []
for k, p in P.items():
    t = m2lib.tabla_umbrales(y, p, CLASES, A["con_alerta"], umbrales, A["costo_no_detectada"], A["costo_falsa_alarma"])
    filas.append(t.assign(modelo=k))
alertas = pd.concat(filas); alertas.round(4).to_csv(RES / "alertas_umbrales.csv", index=False)
tm = alertas[alertas.modelo == mejor].set_index("umbral")
u_opt = tm["costo_por_1000"].idxmin()
umbral = UMBRAL if UMBRAL is not None else A["umbral"]
print(f"umbral de menor costo ({A['costo_no_detectada']}:1) = {u_opt}; umbral configurado = {umbral}")
tm.loc[[0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, u_opt]].round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tm.index, tm["falsas_alarmas_por_1000"], label="falsas alarmas")
ax.plot(tm.index, tm["no_detectadas_por_1000"], label="distracciones no detectadas")
ax.axvline(u_opt, color="k", ls="--", label=f"menor costo ({u_opt})")
ax.set_xlabel("umbral sobre P(con alerta)"); ax.set_ylabel("por cada 1 000 imágenes"); ax.legend(); ax.set_title(mejor)
fig.savefig(FIG / "alertas_umbral.png")

In [ ]:
costos = []
for r in [1, 2, 5, 10, 20]:
    t = m2lib.tabla_umbrales(y, pm, CLASES, A["con_alerta"], umbrales, r, 1.0)
    fila = t.loc[t["costo_por_1000"].idxmin()]
    costos.append({"costo no detectada : falsa alarma": f"{r}:1", "umbral": fila["umbral"],
                   "falsas alarmas por 1 000": fila["falsas_alarmas_por_1000"], "no detectadas por 1 000": fila["no_detectadas_por_1000"]})
pd.DataFrame(costos).round(1)